In [1]:
# ============================================================================
# 奇异曲线攻击 – y^2 = x^3 (判别式 Δ=0), 70-bit 有限域
# 同态映射 phi(P) = x/y (mod p) 将群同构于 F_p 的加法群
# ============================================================================
import time

def add_points(P, Q, p):
    """
    奇异曲线 y^2 = x^3 上的点加法（仿射坐标，不考虑无穷远点）
    公式与标准椭圆曲线相同，因为曲线方程形式一致。
    """
    if P is None: return Q
    if Q is None: return P
    x1, y1 = P
    x2, y2 = Q
    if P == Q:
        # 倍点公式
        if y1 == 0:
            # 切线垂直，返回无穷远点（此处用 None 表示）
            return None
        lam = (3 * x1 * x1) * inverse_mod(2 * y1, p) % p
    else:
        if x1 == x2:
            return None  # 垂直，和为无穷远点
        lam = (y2 - y1) * inverse_mod(x2 - x1, p) % p
    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)

def scalar_mult(k, P, p):
    """标量乘法 k*P (二进制展开法)"""
    result = None
    base = P
    while k:
        if k & 1:
            result = add_points(result, base, p)
        base = add_points(base, base, p)
        k >>= 1
    return result

def phi(point, p):
    """同态映射 phi(P) = x/y (mod p)，将点映射到 F_p 的加法群"""
    if point is None:
        return 0
    x, y = point
    if y == 0:
        # 理论上奇异曲线存在二阶点，但此处我们避免选择 y=0 的点
        raise ValueError("点 y=0 无法映射")
    return x * inverse_mod(y, p) % p

# 1. 生成 70-bit 素数 p
print("=" * 60)
print("奇异曲线攻击 – y^2 = x^3 (Δ=0)")
print("=" * 60)
p = random_prime(2^99, lbound=2^98)
print(f"素数 p = {p} (比特长度: {p.nbits()})")

# 2. 寻找曲线上一个非奇异点（y ≠ 0）作为生成元 P
found = False
while not found:
    x = randint(1, p-1)
    y2 = (x * x * x) % p
    if y2 == 0:
        continue
    y = sqrt(GF(p)(y2))
    if y is not None:
        y = int(y)
        found = True
        P = (x, y)
print(f"生成元 P = {P}")

# 3. 随机生成私钥 k_true，并计算 Q = k_true * P
k_true = randint(2, p-2)   # 避免 0 或 1
print(f"随机私钥 (预期) = {k_true}")

start_time = time.time()
Q = scalar_mult(k_true, P, p)
if Q is None:
    # 如果结果为无穷远点，重新选择私钥
    print("结果出现无穷远点，重新生成私钥")
    k_true = randint(2, p-2)
    Q = scalar_mult(k_true, P, p)
print(f"公钥 Q = {Q}")

# 4. 攻击：利用同态映射求解私钥
phi_P = phi(P, p)
phi_Q = phi(Q, p)
k_found = (phi_Q * inverse_mod(phi_P, p)) % p
elapsed = time.time() - start_time

print(f"求解私钥 = {k_found}")
print(f"奇异曲线攻击破解时间: {elapsed:.6f} 秒")
assert k_found == k_true
print("攻击成功！")

奇异曲线攻击 – y^2 = x^3 (Δ=0)
素数 p = 467125135120609075714975404641 (比特长度: 99)
生成元 P = (356473066859263297540903478728, 149629588317488511831441708133)
随机私钥 (预期) = 380243036863190211292588450806
公钥 Q = (113056825540356110069653047829, 275964283374098271418906135916)
求解私钥 = 380243036863190211292588450806
奇异曲线攻击破解时间: 0.002613 秒
攻击成功！
